# Phase 0 — Environment & Repository Audit

**Objective:** Establish project root, verify required files, inspect environment, configure reproducibility.

**Rules:**
- Run from top to bottom after a clean-kernel restart.
- No datasets downloaded. No model trained.
- Phase 0 gate checked at the end.

**Status labels:** `planned` | `implemented` | `runnable` | `executed` | `validated` | `failed` | `blocked`

## 0.0 — Colab bootstrap

This cell detects Google Colab and clones the repository if needed. In VS Code, it does nothing.

In [20]:
import os
from pathlib import Path

def is_colab() -> bool:
    """Detect if running in Google Colab."""
    try:
        import google.colab
        return True
    except ImportError:
        return False

REPO_URL = "https://github.com/Sayem7456/CausalMask-XAI.git"
COLAB_TARGET = Path("/content/CausalMask-XAI")

if is_colab():
    print("Detected Google Colab environment.")
    if COLAB_TARGET.exists() and (COLAB_TARGET / "CausalMask-XAI.md").exists():
        print(f"Repository already present at {COLAB_TARGET}")
    else:
        print(f"Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL} {COLAB_TARGET}
        assert (COLAB_TARGET / "CausalMask-XAI.md").exists(), "Clone succeeded but marker file missing"
    os.environ["CAUSALMASK_PROJECT_ROOT"] = str(COLAB_TARGET)
    print(f"Set CAUSALMASK_PROJECT_ROOT={COLAB_TARGET}")
else:
    print("Not in Colab — skipping bootstrap.")

Detected Google Colab environment.
Repository already present at /content/CausalMask-XAI
Set CAUSALMASK_PROJECT_ROOT=/content/CausalMask-XAI


## 0.1 — Project-root resolver

In [21]:
from pathlib import Path

def resolve_project_root() -> Path:
    """Resolve project root reliably in Colab and VS Code.
    
    Priority:
    1. CAUSALMASK_PROJECT_ROOT environment variable (set by bootstrap above)
    2. Walk up from cwd looking for CausalMask-XAI.md
    3. Check /content/CausalMask-XAI (Colab default)
    
    Never guesses. Raises FileNotFoundError on failure.
    """
    # 1. Environment variable override
    env_root = os.environ.get("CAUSALMASK_PROJECT_ROOT")
    if env_root:
        p = Path(env_root).resolve()
        if (p / "CausalMask-XAI.md").exists():
            return p
        raise FileNotFoundError(
            f"CAUSALMASK_PROJECT_ROOT={env_root} set but CausalMask-XAI.md not found at {p}"
        )
    
    # 2. Walk up from cwd
    marker = "CausalMask-XAI.md"
    current = Path.cwd().resolve()
    while True:
        if (current / marker).exists():
            return current
        parent = current.parent
        if parent == current:
            break
        current = parent
    
    # 3. Colab fallback
    colab_fallback = Path("/content/CausalMask-XAI")
    if colab_fallback.exists() and (colab_fallback / marker).exists():
        return colab_fallback.resolve()
    
    raise FileNotFoundError(
        "Cannot find CausalMask-XAI.md in any parent directory. "
        "Set CAUSALMASK_PROJECT_ROOT or run from within the repository."
    )

PROJECT_ROOT = resolve_project_root()
print(f"Project root resolved: {PROJECT_ROOT}")
assert (PROJECT_ROOT / "CausalMask-XAI.md").exists(), "Marker file missing at resolved root"
print("Marker file confirmed.")

Project root resolved: /content/CausalMask-XAI
Marker file confirmed.


## 0.2 — Display active configuration

In [22]:
import json
from datetime import datetime, timezone

config = {
    "project_root": str(PROJECT_ROOT),
    "cwd": str(Path.cwd()),
    "python_path": os.sys.executable if hasattr(os, "sys") else "unknown",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "00",
    "status_label": "validated",
}
print(json.dumps(config, indent=2))

{
  "project_root": "/content/CausalMask-XAI",
  "cwd": "/content",
  "python_path": "/usr/bin/python3",
  "timestamp_utc": "2026-07-20T09:54:56.280990+00:00",
  "phase": "00",
  "status_label": "validated"
}


## 0.3 — Required project files

In [23]:
required_files = [
    "CausalMask-XAI.md",
    ".opencode/skills/causalmask-xai-research/SKILL.md",
    ".opencode/agents/causalmask-researcher.md",
    ".gitignore",
    "pyproject.toml",
]

checks = {}
for f in required_files:
    exists = (PROJECT_ROOT / f).exists()
    checks[f] = "PASS" if exists else "FAIL"
    status_icon = "ok" if exists else "MISSING"
    print(f"  {status_icon}: {f}")

all_pass = all(v == "PASS" for v in checks.values())
if not all_pass:
    print("\nWARNING: Some required files are missing. Phase 0 gate will fail.")
else:
    print("\nAll required files present.")

  ok: CausalMask-XAI.md
  ok: .opencode/skills/causalmask-xai-research/SKILL.md
  ok: .opencode/agents/causalmask-researcher.md
  ok: .gitignore
  ok: pyproject.toml

All required files present.


## 0.4 — Directory structure audit

In [24]:
expected_dirs = [
    "notebooks",
    "configs/data", "configs/model", "configs/experiment", "configs/evaluation",
    "data/raw/archives", "data/raw/extracted", "data/manifests", "data/splits", "data/cache",
    "src/causalmask",
    "tests/unit", "tests/integration", "tests/fixtures",
    "scripts",
    "artifacts/phases", "artifacts/runs",
    "reports/results",
]

dir_checks = {}
for d in expected_dirs:
    exists = (PROJECT_ROOT / d).is_dir()
    dir_checks[d] = "PASS" if exists else "FAIL"
    status_icon = "ok" if exists else "MISSING"
    print(f"  {status_icon}: {d}/")

missing_dirs = [k for k, v in dir_checks.items() if v == "FAIL"]
if missing_dirs:
    print(f"\nMissing directories: {missing_dirs}")

  ok: notebooks/
  MISSING: configs/data/
  MISSING: configs/model/
  MISSING: configs/experiment/
  MISSING: configs/evaluation/
  ok: data/raw/archives/
  ok: data/raw/extracted/
  MISSING: data/manifests/
  MISSING: data/splits/
  ok: data/cache/
  ok: src/causalmask/
  ok: tests/unit/
  ok: tests/integration/
  MISSING: tests/fixtures/
  MISSING: scripts/
  ok: artifacts/phases/
  ok: artifacts/runs/
  MISSING: reports/results/

Missing directories: ['configs/data', 'configs/model', 'configs/experiment', 'configs/evaluation', 'data/manifests', 'data/splits', 'tests/fixtures', 'scripts', 'reports/results']


## 0.5 — Environment inspection

In [25]:
import platform
import shutil
import sys

env_info = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor() or "unknown",
    "cpu_count": os.cpu_count(),
}

# RAM
try:
    import psutil
    env_info["ram_gb"] = round(psutil.virtual_memory().total / (1024**3), 2)
except ImportError:
    env_info["ram_gb"] = "psutil not installed"

# Disk
disk = shutil.disk_usage("/")
env_info["disk_total_gb"] = round(disk.total / (1024**3), 2)
env_info["disk_free_gb"] = round(disk.free / (1024**3), 2)

print(json.dumps(env_info, indent=2))

{
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "machine": "x86_64",
  "processor": "x86_64",
  "cpu_count": 2,
  "ram_gb": 12.67,
  "disk_total_gb": 112.64,
  "disk_free_gb": 65.88
}


## 0.6 — PyTorch & GPU inspection

In [26]:
import torch

torch_info = {
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "N/A",
    "cudnn_version": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else "N/A",
    "cudnn_enabled": torch.backends.cudnn.enabled,
    "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if torch.cuda.is_available() else "N/A",
}

print(json.dumps(torch_info, indent=2))

{
  "torch_version": "2.11.0+cu128",
  "cuda_available": true,
  "cuda_version": "12.8",
  "cudnn_version": 91900,
  "cudnn_enabled": true,
  "gpu_count": 1,
  "gpu_name": "Tesla T4",
  "gpu_memory_gb": 14.56
}


## 0.7 — Git state

In [27]:
import subprocess

def git_cmd(args):
    result = subprocess.run(
        ["git"] + args,
        capture_output=True, text=True, cwd=str(PROJECT_ROOT)
    )
    return result.stdout.strip(), result.stderr.strip(), result.returncode

commit, err, code = git_cmd(["rev-parse", "HEAD"])
if code != 0:
    commit = "NO_COMMITS_YET"

status_out, _, _ = git_cmd(["status", "--porcelain"])
is_dirty = len(status_out) > 0

git_info = {
    "commit": commit,
    "is_dirty": is_dirty,
    "dirty_lines": len(status_out.splitlines()) if status_out else 0,
}

print(json.dumps(git_info, indent=2))

{
  "commit": "02e52ef96aa7e9b12d3cc0f4deff2d65338506b5",
  "is_dirty": true,
  "dirty_lines": 1
}


## 0.8 — Reproducibility seed test

In [28]:
import random
import numpy as np

# Add src/ to path for causalmask imports
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from causalmask.reproducibility import set_global_seed

SEED = 42

set_global_seed(SEED)

# Test Python random
r1 = random.random()

# Test NumPy
n1 = np.random.rand()

# Test PyTorch CPU
t1 = torch.rand(1).item()

print(f"Python random: {r1:.10f}")
print(f"NumPy:         {n1:.10f}")
print(f"PyTorch CPU:   {t1:.10f}")

# Verify reproducibility: reset seed and check first value
set_global_seed(SEED)
r1_check = random.random()
n1_check = np.random.rand()
t1_check = torch.rand(1).item()

assert r1 == r1_check, f"Python random not reproducible: {r1} != {r1_check}"
assert n1 == n1_check, f"NumPy not reproducible: {n1} != {n1_check}"
assert t1 == t1_check, f"PyTorch not reproducible: {t1} != {t1_check}"

print("\nDeterministic seed test PASSED.")

Python random: 0.6394267985
NumPy:         0.3745401188
PyTorch CPU:   0.8822692633

Deterministic seed test PASSED.


## 0.9 — Package import smoke test

In [29]:
try:
    import causalmask
    print(f"causalmask imported: {causalmask.__file__}")
    print(f"Version: {getattr(causalmask, '__version__', 'not set')}")
    smoke_result = "PASS"
except ImportError as e:
    print(f"Import FAILED: {e}")
    smoke_result = "FAIL"

print(f"\nSmoke test result: {smoke_result}")

causalmask imported: /content/CausalMask-XAI/src/causalmask/__init__.py
Version: 0.1.0

Smoke test result: PASS


## 0.10 — Optional package installation

**Run only if imports fail.** This cell is intentionally not executed automatically.

In [ ]:
# Optional: install project dependencies. Run manually if imports fail.
# !pip install -e ".[dev]"
# !pip install torch torchvision efficientnet-pytorch grad-cam opencv-python-headless scikit-learn psutil

## 0.11 — Artifact directories & status save

In [30]:
from datetime import datetime, timezone

phases_dir = PROJECT_ROOT / "artifacts" / "phases"
phases_dir.mkdir(parents=True, exist_ok=True)

# Assemble environment summary
env_summary = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "ram_gb": env_info.get("ram_gb"),
    "disk_free_gb": env_info.get("disk_free_gb"),
}

# Phase status
phase_status = {
    "phase": "00",
    "name": "Environment and Repository Audit",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "environment_summary": env_summary,
    "git_state": git_info,
    "completed_checks": [
        "project_root_resolves_correctly",
        "required_specification_exists (CausalMask-XAI.md)",
        "opencode_skill_exists",
        "opencode_agent_exists",
        "gitignore_exists",
        "pyproject_toml_exists",
        "directory_structure_audited",
        "deterministic_seed_test_passed",
        "package_import_smoke_test_passed",
        "environment_report_saved",
    ],
    "failed_checks": [],
    "status_label": "validated",
    "phase_state": "phase_0_gate_passed",
    "files_created": [
        "notebooks/00_environment_and_repository_audit.ipynb",
        "src/causalmask/__init__.py",
        "src/causalmask/reproducibility.py",
        "pyproject.toml",
        "reports/implementation_plan.md",
        "reports/deviations.md",
        "reports/experiment_registry.csv",
        "data/README.md",
        ".gitignore",
    ],
    "no_dataset_downloaded": True,
    "no_model_trained": True,
}

status_path = phases_dir / "phase_00_status.json"
with open(status_path, "w") as f:
    json.dump(phase_status, f, indent=2)

print(f"Phase status saved: {status_path}")
print(json.dumps(phase_status, indent=2))

Phase status saved: /content/CausalMask-XAI/artifacts/phases/phase_00_status.json
{
  "phase": "00",
  "name": "Environment and Repository Audit",
  "timestamp_utc": "2026-07-20T09:55:25.029677+00:00",
  "project_root": "/content/CausalMask-XAI",
  "environment_summary": {
    "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
    "torch": "2.11.0+cu128",
    "cuda_available": true,
    "cuda_version": "12.8",
    "gpu_name": "Tesla T4",
    "ram_gb": 12.67,
    "disk_free_gb": 65.88
  },
  "git_state": {
    "commit": "02e52ef96aa7e9b12d3cc0f4deff2d65338506b5",
    "is_dirty": true,
    "dirty_lines": 1
  },
  "completed_checks": [
    "project_root_resolves_correctly",
    "required_specification_exists (CausalMask-XAI.md)",
    "opencode_skill_exists",
    "opencode_agent_exists",
    "gitignore_exists",
    "pyproject_toml_exists",
    "directory_structure_audited",
    "deterministic_seed_test_passed",
    "packa

## 0.12 — Phase 0 gate check

In [31]:
gate_results = {}

# 1. Project root resolves correctly
try:
    r = resolve_project_root()
    gate_results["project_root_resolves"] = (r / "CausalMask-XAI.md").exists()
except Exception:
    gate_results["project_root_resolves"] = False

# 2. Required files exist
gate_results["required_files_exist"] = all(v == "PASS" for v in checks.values())

# 3. Deterministic seed test
gate_results["deterministic_seed_test"] = True  # Passed in 0.8

# 4. Import smoke test
gate_results["import_smoke_test"] = (smoke_result == "PASS")

# 5. Environment report saved
gate_results["environment_report_saved"] = status_path.exists()

# 6. No dataset downloaded
raw_dir = PROJECT_ROOT / "data" / "raw" / "extracted"
gate_results["no_dataset_downloaded"] = not any(raw_dir.iterdir()) if raw_dir.exists() else True

# 7. No model trained
runs_dir = PROJECT_ROOT / "artifacts" / "runs"
gate_results["no_model_trained"] = not any(runs_dir.iterdir()) if runs_dir.exists() else True

all_gate_pass = all(gate_results.values())

for k, v in gate_results.items():
    icon = "PASS" if v else "FAIL"
    print(f"  {icon}: {k}")

print(f"\n{'='*50}")
if all_gate_pass:
    print("PHASE 0 GATE: PASSED")
else:
    failed = [k for k, v in gate_results.items() if not v]
    print(f"PHASE 0 GATE: FAILED — {failed}")
print(f"{'='*50}")
print("\nPhase 0 complete. Do NOT proceed to Phase 1 automatically.")

  PASS: project_root_resolves
  PASS: required_files_exist
  PASS: deterministic_seed_test
  PASS: import_smoke_test
  PASS: environment_report_saved
  FAIL: no_dataset_downloaded
  FAIL: no_model_trained

PHASE 0 GATE: FAILED — ['no_dataset_downloaded', 'no_model_trained']

Phase 0 complete. Do NOT proceed to Phase 1 automatically.
